# Vesuvius rite of passage — 2.5D U-Net ink-detection baseline

**This is practice, not the August submission.** It exists to produce *your first ink image*
and to teach the terrain: surface volumes, masks, labels, patches, and inference stitching.

## Run it on Kaggle (free GPU)

1. **kaggle.com → Create → New Notebook**
2. Right sidebar → **Session options → Accelerator → GPU** (T4 x2 or P100).
   Without this the training loop is unusably slow — the code will warn you.
3. Right sidebar → **Input → Add Input** → search `vesuvius-challenge-ink-detection`
   → add the competition dataset. It mounts at
   `/kaggle/input/vesuvius-challenge-ink-detection/`.
4. **Run All.** Fragment 1 at `z_dim=16` needs roughly 6–8 GB of RAM to hold the
   volume; 8 epochs takes on the order of 15–30 minutes on a T4.

## What success looks like

`out/prediction.png` shows ghostly bright strokes where the model believes there is ink,
on a vertical strip it never trained on. Compare it against `out/ground_truth.png`.

Dependencies (`torch`, `numpy`, `pillow`, `tqdm`) are all preinstalled on Kaggle.

## 1. Imports

In [ ]:
import os
import random
import numpy as np
from PIL import Image

Image.MAX_IMAGE_PIXELS = None  # fragments are huge; disable PIL's bomb check

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm import tqdm
except ImportError:  # tqdm is nice, not required
    def tqdm(x, **k):
        return x

## 2. Config

The only block you should need to touch. `data_dir` already points at the Kaggle mount path.

In [ ]:
CFG = {
    "data_dir": "/kaggle/input/vesuvius-challenge-ink-detection/train",
    "fragment": "1",     # "1" is the classic starter fragment
    "z_start": 24,       # the middle slices carry most of the ink signal
    "z_dim": 16,         # how many slices the model sees (input channels)
    "patch": 224,        # training patch size (pixels)
    "stride": 112,       # patch grid stride (overlapping patches)
    "batch_size": 16,
    "epochs": 8,
    "lr": 1e-4,
    "val_frac": 0.15,    # rightmost vertical strip is held out for validation
    "mask_min": 0.05,    # keep patches with at least this much papyrus
    "out_dir": "out",
    "seed": 42,
}

## 3. Reproducibility and device

Fixed seed (42). If this prints the CPU warning, go back and switch the accelerator to GPU.

In [ ]:
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU found — this will be painfully slow. "
          "On Kaggle: Settings -> Accelerator -> GPU.")

## 4. Data loading

`load_fragment` reads the mask, the ink labels, and a `z_dim`-slice window of the surface volume. `build_patch_grid` enumerates every patch corner whose patch holds at least `mask_min` papyrus.

In [ ]:
def load_fragment(frag_dir, z_start, z_dim):
    """Return (volume[z,H,W] float16 in 0..1, mask[H,W] bool, labels[H,W] float32)."""
    mask = np.array(Image.open(os.path.join(frag_dir, "mask.png")).convert("L")) > 0
    labels = (np.array(Image.open(os.path.join(frag_dir, "inklabels.png")).convert("L")) > 0)
    labels = labels.astype(np.float32)

    slices = []
    for z in tqdm(range(z_start, z_start + z_dim), desc="loading slices"):
        p = os.path.join(frag_dir, "surface_volume", f"{z:02}.tif")
        arr = np.array(Image.open(p), dtype=np.float32) / 65535.0
        slices.append(arr.astype(np.float16))  # float16 to fit in RAM
    volume = np.stack(slices)  # (z_dim, H, W)
    return volume, mask, labels


def build_patch_grid(mask, patch, stride, mask_min):
    """All (y, x) top-left corners whose patch contains enough papyrus."""
    H, W = mask.shape
    coords = []
    for y in range(0, H - patch + 1, stride):
        for x in range(0, W - patch + 1, stride):
            if mask[y:y + patch, x:x + patch].mean() >= mask_min:
                coords.append((y, x))
    return coords


class PatchDataset(Dataset):
    def __init__(self, volume, labels, coords, patch, augment):
        self.volume, self.labels = volume, labels
        self.coords, self.patch, self.augment = coords, patch, augment

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, i):
        y, x = self.coords[i]
        p = self.patch
        img = self.volume[:, y:y + p, x:x + p].astype(np.float32)   # (C,H,W)
        lab = self.labels[y:y + p, x:x + p][None]                    # (1,H,W)
        if self.augment:
            k = random.randint(0, 3)
            if k:
                img = np.rot90(img, k, axes=(1, 2))
                lab = np.rot90(lab, k, axes=(1, 2))
            if random.random() < 0.5:
                img = np.flip(img, axis=2)
                lab = np.flip(lab, axis=2)
        return torch.from_numpy(img.copy()), torch.from_numpy(lab.copy())

## 5. Model — a compact U-Net

Input channels = `z_dim` slices, output = a single ink logit per pixel. The z-axis is folded into the channel dimension: that is what makes this “2.5D” rather than true 3D.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, c_in, base=32):
        super().__init__()
        self.d1 = DoubleConv(c_in, base)
        self.d2 = DoubleConv(base, base * 2)
        self.d3 = DoubleConv(base * 2, base * 4)
        self.d4 = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.u3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.u2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.u1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        s1 = self.d1(x)
        s2 = self.d2(self.pool(s1))
        s3 = self.d3(self.pool(s2))
        b = self.d4(self.pool(s3))
        x = self.u3(torch.cat([self.up3(b), s3], dim=1))
        x = self.u2(torch.cat([self.up2(x), s2], dim=1))
        x = self.u1(torch.cat([self.up1(x), s1], dim=1))
        return self.head(x)

## 6. Metric

In [ ]:
def dice_score(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    pred = (prob > 0.5).float()
    inter = (pred * target).sum()
    return ((2 * inter + eps) / (pred.sum() + target.sum() + eps)).item()

## 7. Load the fragment and build the train/validation split

The rightmost vertical strip (15% of the width) is held out. Splitting by position rather than at random matters here: neighbouring patches overlap, so a random split would leak ink from a training patch straight into validation.

In [ ]:
os.makedirs(CFG["out_dir"], exist_ok=True)
frag_dir = os.path.join(CFG["data_dir"], CFG["fragment"])

volume, mask, labels = load_fragment(frag_dir, CFG["z_start"], CFG["z_dim"])
H, W = mask.shape
split_x = int(W * (1 - CFG["val_frac"]))  # right strip = validation
print(f"fragment {CFG['fragment']}: {H}x{W}, validation strip starts at x={split_x}")

coords = build_patch_grid(mask, CFG["patch"], CFG["stride"], CFG["mask_min"])
train_c = [c for c in coords if c[1] + CFG["patch"] <= split_x]
val_c = [c for c in coords if c[1] >= split_x]
print(f"patches: {len(train_c)} train / {len(val_c)} val")

train_dl = DataLoader(PatchDataset(volume, labels, train_c, CFG["patch"], True),
                      batch_size=CFG["batch_size"], shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl = DataLoader(PatchDataset(volume, labels, val_c, CFG["patch"], False),
                    batch_size=CFG["batch_size"], shuffle=False,
                    num_workers=2, pin_memory=True)

## 8. Train

Best checkpoint by validation Dice is saved to `out/best.pt`.

In [ ]:
model = UNet(CFG["z_dim"]).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
loss_fn = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

best_dice = 0.0
for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    tr_loss = 0.0
    for img, lab in tqdm(train_dl, desc=f"epoch {epoch} train"):
        img, lab = img.to(DEVICE), lab.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            loss = loss_fn(model(img), lab)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tr_loss += loss.item()

    model.eval()
    va_loss, va_dice, n = 0.0, 0.0, 0
    with torch.no_grad():
        for img, lab in val_dl:
            img, lab = img.to(DEVICE), lab.to(DEVICE)
            logits = model(img)
            va_loss += loss_fn(logits, lab).item()
            va_dice += dice_score(logits, lab)
            n += 1
    va_loss, va_dice = va_loss / max(n, 1), va_dice / max(n, 1)
    print(f"epoch {epoch}: train_loss={tr_loss/len(train_dl):.4f} "
          f"val_loss={va_loss:.4f} val_dice={va_dice:.4f}")
    if va_dice > best_dice:
        best_dice = va_dice
        torch.save(model.state_dict(), os.path.join(CFG["out_dir"], "best.pt"))
        print(f"  ^ new best (dice {best_dice:.4f}) — saved")

## 9. Inference — sliding window over the held-out strip

Overlapping windows are averaged (`acc / cnt`), then everything outside the papyrus mask is blanked. This is the cell that produces your ink image.

In [ ]:
# Inference: sliding window over the validation strip -> ink image
print("running sliding-window inference on the validation strip...")
model.load_state_dict(torch.load(os.path.join(CFG["out_dir"], "best.pt")))
model.eval()
p, s = CFG["patch"], CFG["stride"]
acc = np.zeros((H, W - split_x), dtype=np.float32)
cnt = np.zeros_like(acc)
with torch.no_grad():
    for y in tqdm(range(0, H - p + 1, s), desc="inference"):
        batch, spots = [], []
        for x in range(split_x, W - p + 1, s):
            batch.append(volume[:, y:y + p, x:x + p].astype(np.float32))
            spots.append(x - split_x)
        if not batch:
            continue
        t = torch.from_numpy(np.stack(batch)).to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            prob = torch.sigmoid(model(t)).squeeze(1).float().cpu().numpy()
        for pr, x0 in zip(prob, spots):
            acc[y:y + p, x0:x0 + p] += pr
            cnt[y:y + p, x0:x0 + p] += 1

pred = np.divide(acc, cnt, out=np.zeros_like(acc), where=cnt > 0)
pred *= mask[:, split_x:]  # blank out non-papyrus
Image.fromarray((pred * 255).astype(np.uint8)).save(
    os.path.join(CFG["out_dir"], "prediction.png"))
Image.fromarray((labels[:, split_x:] * 255).astype(np.uint8)).save(
    os.path.join(CFG["out_dir"], "ground_truth.png"))
print(f"DONE. Open {CFG['out_dir']}/prediction.png — those bright strokes "
      "are 2,000-year-old ink, found by YOUR model. Post it.")

## 10. Look at it

Run the cell below to display the prediction next to the ground truth inside the notebook,
then **download `out/prediction.png` and post it**. Those bright strokes are 2,000-year-old
ink, found by a model you trained.

> This cell is the one addition to the original script — it only *displays* the two PNGs
> the script already wrote. It changes nothing about the training or inference logic.

In [ ]:
import matplotlib.pyplot as plt

pred_img = Image.open(os.path.join(CFG["out_dir"], "prediction.png"))
gt_img = Image.open(os.path.join(CFG["out_dir"], "ground_truth.png"))

fig, axes = plt.subplots(1, 2, figsize=(14, 10))
axes[0].imshow(pred_img, cmap="gray")
axes[0].set_title("prediction (never trained on this strip)")
axes[0].axis("off")
axes[1].imshow(gt_img, cmap="gray")
axes[1].set_title("ground truth")
axes[1].axis("off")
plt.tight_layout()
plt.show()